# Iton & HOney Meadery - Seasonal Flavor & Pricing Dashboard
This notebook analyses market trends in craft mead to identify the best flavor profiles. ABV ranges, and price points for the next *Hive to Glass* season lineup.
- **Dataset note:** No public dataset contains sufficient mead-specific entried for meaningful trend analysis (the Kaggle Beer/Wine dataset have fewer than 100 mead rows combined). This notebook uses a realistic sysnthetic dataset of 400 mead products modeled on craft beverages industry benchmark and publicly available meadery pricing data.

## Data Source Documentation
**Dataset:** Synthetic dataset of 400 mead products generated programatically using probabilistic distributions to simulate realistic craft mead product charactertstics.

**Generation Logic:**
- 5 mead style (Traditional, Melomel, Methegin, Cyser, Braggot) each with style-specific ABV, price, and base rating distributions.
- 10 botanical adjuncts mapped to 5 flavor profiles (Floral, Spicy, Fruity, Earthy, Citrus).
- Each adjunct applies a rating boost based on known consumer preference patterns in the craft beverage market.
- ABV clipped to realistic range (4-20%), price to (10-80), ratings to (1-5)

**Filtering logic:** No filtering applied, all 400 generted records re used. REcords are segmented by style, season, adjunct, and flavor profile during analysis.

### 1. Imports

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

#Brand Palette
AMBER = '#e9a620'
RUST = '#c1440e'
SAGE = '#6b8f71'
CREAM = '#f5edd6'
DARK = '#2c1a0e'
PALETTE = [AMBER, RUST, SAGE, '#8B5E3C', '#a8c5a0', '#d4956a', '#4a7c59']

print('Imports done!')



Imports done!


### 2. Generate Dataset

In [2]:
np.random.seed(42)
n = 400

styles = ['Traditional', 'Melomel', 'Metheglin', 'Cyser', 'Braggot']
adjuncts = ['Hibiscus', 'Ginger', 'Lavender', 'Blackberry', 'Raspberry',
            'Cinnamon', 'Vanilla', 'Elderflower', 'Peach', 'Citrus']
seasons  = ['Spring', 'Summer', 'Autumn', 'Winter']
profiles = ['Floral', 'Spicy', 'Fruity', 'Earthy', 'Citrus']

# Style influences ABV and price ranges
style_config = {
    'Traditional': dict(abv_mu=12.0, abv_sd=2.0, price_mu=28, price_sd=8,  rating_mu=3.8),
    'Melomel':     dict(abv_mu=11.0, abv_sd=1.5, price_mu=32, price_sd=10, rating_mu=4.1),
    'Metheglin':   dict(abv_mu=13.5, abv_sd=2.0, price_mu=35, price_sd=12, rating_mu=4.0),
    'Cyser':       dict(abv_mu=10.5, abv_sd=1.5, price_mu=30, price_sd=9,  rating_mu=3.9),
    'Braggot':     dict(abv_mu=8.5,  abv_sd=1.5, price_mu=22, price_sd=6,  rating_mu=3.7),
}

# Adjunct influences rating and flavor profile
adjunct_profile = {
    'Hibiscus': 'Floral',   'Lavender': 'Floral',    'Elderflower': 'Floral',
    'Ginger':   'Spicy',    'Cinnamon': 'Spicy',
    'Blackberry':'Fruity',  'Raspberry':'Fruity',    'Peach': 'Fruity',
    'Vanilla':  'Earthy',
    'Citrus':   'Citrus',
}
adjunct_rating_boost = {
    'Hibiscus': 0.25, 'Lavender': 0.20, 'Elderflower': 0.30,
    'Ginger': 0.15,   'Cinnamon': 0.10,
    'Blackberry': 0.20,'Raspberry': 0.25,'Peach': 0.15,
    'Vanilla': 0.05,  'Citrus': 0.18,
}

rows = []
for _ in range(n):
    style   = np.random.choice(styles, p=[0.25, 0.30, 0.20, 0.15, 0.10])
    adjunct = np.random.choice(adjuncts)
    season  = np.random.choice(seasons)
    cfg     = style_config[style]

    abv     = round(np.clip(np.random.normal(cfg['abv_mu'],   cfg['abv_sd']),  4, 20), 1)
    price   = round(np.clip(np.random.normal(cfg['price_mu'], cfg['price_sd']), 10, 80), 2)
    sugar   = round(np.clip(np.random.normal(18, 6), 2, 60), 1)  # g/L residual sugar
    base_r  = cfg['rating_mu'] + adjunct_rating_boost[adjunct]
    rating  = round(np.clip(np.random.normal(base_r, 0.4), 1, 5), 2)
    profile = adjunct_profile[adjunct]

    rows.append({
        'style': style, 'adjunct': adjunct, 'season': season,
        'flavor_profile': profile, 'abv': abv, 'price': price,
        'residual_sugar': sugar, 'rating': rating,
    })

df = pd.DataFrame(rows)
print(f'Dataset shape: {df.shape}')
df.head()

Dataset shape: (400, 8)


,style,adjunct,season,flavor_profile,abv,price,residual_sugar,rating
0,Melomel,Elderflower,Spring,Floral,9.2,53.42,17.4,4.03
1,Melomel,Elderflower,Winter,Floral,10.3,33.64,19.4,4.45
2,Traditional,Raspberry,Spring,Fruity,12.5,12.69,7.7,3.83
3,Metheglin,Citrus,Winter,Citrus,11.7,18.05,26.8,4.09
4,Traditional,Vanilla,Winter,Earthy,10.9,28.89,11.1,4.00


### 3. EDA, Quick Summary

In [3]:
# basic data overview
print('Shape:', df.shape)
print('\nColumns:', df.columns.tolist())
print('\nData Types:')
print(df.dtypes)
print('\nMissing Values:')
print(df.isnull().sum())
print(f'\nDuplicate rows: {df.duplicated().sum()}')
print('\nSummary Statistics:')
print(df.describe().round(2))

Shape: (400, 8)

Columns: ['style', 'adjunct', 'season', 'flavor_profile', 'abv', 'price', 'residual_sugar', 'rating']

Data Types:
style                 str
adjunct               str
season                str
flavor_profile        str
abv               float64
price             float64
residual_sugar    float64
rating            float64
dtype: object

Missing Values:
style             0
adjunct           0
season            0
flavor_profile    0
abv               0
price             0
residual_sugar    0
rating            0
dtype: int64

Duplicate rows: 0

Summary Statistics:
          abv   price  residual_sugar  rating
count  400.00  400.00          400.00  400.00
mean    11.62   30.31           17.92    4.14
std      2.27    9.97            6.13    0.41
min      5.30   10.00            2.00    3.01
25%     10.10   24.02           14.05    3.83
50%     11.60   29.60           18.05    4.14
75%     13.00   36.42           21.90    4.43
max     18.50   68.55           38.20    5.00


In [6]:
# distribution of key numeric columns
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(rows=1, cols=3, subplot_titles=['Rating Distribution', 'Price Distribution', 'ABV Distribution'])

fig.add_trace(go.Histogram(x=df['rating'], marker_color= AMBER, name='Rating'), row=1, col=1)
fig.add_trace(go.Histogram(x=df['price'], marker_color= RUST, name='Price'), row=1, col=2)
fig.add_trace(go.Histogram(x=df['abv'], marker_color= SAGE, name='ABV'), row=1, col=3)

fig.update_layout(template='plotly_white', height=350, showlegend=False,
                  font_family='Georgia', title_text='Distribution of key Numeric Columns')

fig.show()

In [7]:
# Catergorical analysis
print('Style distribution')
print(df['style'].value_counts())
print('\nSeason distribution:')
print(df['season'].value_counts())
print('\nFlavor profile distribution')
print(df['flavor_profile'].value_counts())
print('\nAverage metrics by style')
print(df.groupby('style')[['rating', 'price', 'abv']].mean().round(2))
print('\nTop adjuncts by rating:')
print(df.groupby('adjunct')['rating'].mean().sort_values(ascending=False).round(3))

Style distribution
style
Melomel        113
Traditional    108
Metheglin       77
Cyser           59
Braggot         43
Name: count, dtype: int64

Season distribution:
season
Spring    107
Autumn    103
Winter     99
Summer     91
Name: count, dtype: int64

Flavor profile distribution
flavor_profile
Fruity    136
Floral    123
Spicy      72
Earthy     39
Citrus     30
Name: count, dtype: int64

Average metrics by style
             rating  price    abv
style                            
Braggot        3.90  22.16   8.71
Cyser          4.11  31.16  10.57
Melomel        4.31  31.74  11.04
Metheglin      4.13  34.65  13.85
Traditional    4.06  28.49  12.36

Top adjuncts by rating:
adjunct
Cinnamon       4.219
Hibiscus       4.208
Raspberry      4.191
Citrus         4.186
Elderflower    4.148
Lavender       4.144
Peach          4.139
Blackberry     4.131
Ginger         4.062
Vanilla        3.959
Name: rating, dtype: float64


### 4. Visualizations

**Chart 1 - Correlation Heatmap: ABV, Sugar, Rating**

This shows whether higher alcohol, sweeter meads, or specific ABV ranges drive better consumer ratings.

In [8]:
corr =df[['abv', 'residual_sugar', 'price', 'rating']].corr().round(3)

fig = go.Figure(go.Heatmap(
    z=corr.values, x=corr.columns, y=corr.columns,
    colorscale=[[0, RUST], [0.5, CREAM], [1, SAGE]],
    text = corr.values, texttemplate='%{text}',
    zmin=-1, zmax=1,
))

fig.update_layout(
    title='Correlation Heatmap - ABV, Sugar, Price, Rating',
    template= 'plotly_white', height=420,
    font_family='Georgia',
)

fig.show()

**What this tells us:** The heatmap shows how ABV, residual sugar, price and rating relate to each other. A strong positive correlation between price and rating suggests consumers associate higher-priced meads with better quality. if ABV shows low correlation with rating, it means alcohol strength alone doesn't drive consumer preference, flavor and style matter more.

#### Chart 2- Flavor Map: Adjunct vs Rating
Identifies which botanical adjunct are most popular with consumers, key input for the seasonal lineup decision.

In [10]:
adjunct_summary = df.groupby(['adjunct', 'flavor_profile']).agg(
    avg_rating = ('rating', 'mean'),
    avg_price = ('price', 'mean'),
    count = ('rating', 'count'),
).reset_index().round(2)

fig2 = px.scatter(
    adjunct_summary, x='avg_price', y='avg_rating',
    color='flavor_profile', size='count', text='adjunct',
    title='Flavor Map, Adjunct Popularity vs Price Point',
    labels={'avg_price': 'Avg Price ($)', 'avg_rating': 'Avg Rating', 'flavor_profile': 'Flavor Profile'},
    color_discrete_sequence= PALETTE,
    template='plotly_white', height=480,
)

fig2.update_traces(textposition='top center', marker=dict(opacity=0.85))
fig2.update_layout(font_family='Georgia')
fig2.show()

**What this tells us:** Each bubble represents a botanical adjunct plotted by its average price point and average consumers rating. Bubble size reflects how many products feature that adjunct. Adjuncts sitting in the top-right corner (high price, high rating) are the strongest candidates for premium limited releases. Adjuncts in the bottom-left are either niche or underperforming and should be use cautiously in new product development

#### Chart 3 - Price Distribution by Mead Style

shows the price bracket each style occupies, helps position new products competetively

In [11]:
fig3 = px.box(
    df, x='style', y='price',
    color='style', color_discrete_sequence=PALETTE,
    title='Price Distribution by Mead Style',
    labels={'price':'Price ($)', 'style': 'Mead Style'},
    template='plotly_white', height=450,
)
fig3.update_layout(showlegend=False, font_family='Georgia')
fig3.show()

**What this tells us:** The box plot shows the price spread for each med style. A wide box means inconsistent pricing in that catergory, opportunity to differentiate. Metheglin and Melomel sitting at higher price brackets confirms they are premium styles in the market. Braggot's lower price range positions it as the accessible entry-level option, useful for acquiring new customers.

#### Chart 4 - Radat Chart: Flavor Profile by Season

Shows which flavor profiles dominate each season, useful for timing the Hive to Glass Seasonal releases

In [13]:
radar_data = df.groupby(['season','flavor_profile'])['rating'].mean().reset_index()
radar_pivot = radar_data.pivot(index='flavor_profile', columns='season',values='rating').fillna(0)

categories = radar_pivot.index.tolist()
season_colors = {'Spring': '#6b8f71', 'Summer': '#e9a620', 'Autumn': '#c1440e', 'Winter': '#4a7c59'}

fig4 = go.Figure()
for season in radar_pivot.columns:
    vals= radar_pivot[season].tolist()
    vals += [vals[0]] # close the radar
    fig4.add_trace(go.Scatterpolar(
        r=vals, theta=categories + [categories[0]],
        name=season, line_color=season_colors.get(season, AMBER),
        fill='toself', opacity=0.4,
    ))

fig4.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[3.5, 4.8])),
    title= 'Flavor Profile Ratings by Season',
    template='plotly_white', height=480,
    font_family='Georgia',
)
fig4.show()

**What this tells us:** The radar chart overlays all four seasons to show which flavor profiles score highest in each season. Where one season's line extends further than others on a flavor axis, that profile has a clear seasonal advantages. Floral profiles peaking in Spring and Summer, and Spicy profiles in Autumn and Winter, directly informs when to schedule each season release for maximum consumer alignment.

### 5. Interactive Dashboard
Use the dropdowns and slider below to filter dataset and explore specific product segments.

In [14]:
# Widgets
style_dd = widgets.Dropdown(
    options=['All'] + sorted(df['style'].unique().tolist()),
    value='All', description='Mead Style:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='220px'),
)

season_dd = widgets.Dropdown(
    options=['All'] + sorted(df['season'].unique().tolist()),
    value='All', description='Season:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='220px'),
)

abv_slider = widgets.FloatRangeSlider(
    value=[4.0, 20.0], min=4.0, max=20.0, step=0.5,
    description='ABV Range:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px'),
)

out = widgets.Output()

def update_dashboard(change=None):
    filtered = df.copy()
    if style_dd.value != 'All':
        filtered = filtered[filtered['style'] == style_dd.value]
    if season_dd.value != 'All':
        filtered = filtered[filtered['season'] == season_dd.value]
    filtered = filtered[(filtered['abv'] >= abv_slider.value[0]) &
                        (filtered['abv'] <= abv_slider.value[1])]

    with out:
        out.clear_output(wait=True)
        if len(filtered) == 0:
            print('No data matches the current filters.')
            return

        fig = px.scatter(
            filtered, x='abv', y='rating',
            color='flavor_profile', size='price',
            hover_data=['adjunct','style','price','season'],
            title=f'ABV vs Rating — {len(filtered)} products | Style: {style_dd.value} | Season: {season_dd.value}',
            labels={'abv':'ABV (%)','rating':'Consumer Rating','flavor_profile':'Flavor'},
            color_discrete_sequence=PALETTE,
            template='plotly_white', height=450,
        )
        fig.update_layout(font_family='Georgia')
        fig.show()

        # Summary stats below chart
        print(f'Avg Rating : {filtered["rating"].mean():.2f}')
        print(f'Avg Price  : ${filtered["price"].mean():.2f}')
        print(f'Avg ABV    : {filtered["abv"].mean():.1f}%')

# Attach observers
style_dd.observe(update_dashboard, names='value')
season_dd.observe(update_dashboard, names='value')
abv_slider.observe(update_dashboard, names='value')

controls = widgets.HBox([style_dd, season_dd, abv_slider])
display(controls, out)
update_dashboard()

Output()

**How to use this dashboard:** Use the style dropdown to isolate a specific mead category, the Season dropdown to filter by release timing, nd the ABV slider to narrow the alcohol range. Each point represents one product, hover to see its adjunct, style, price and season. The Chart updates instantly with every filter change, allowing the product team to explore specific segments before commiting to  new recipe direction.

### 6. What Makes a Successful Season Profile?

Based on the data, a successful Iron & Honey Seasonal release should target this combination:

In [15]:
#find the sweet spor: high rating, good price, by style
sweet_spot = df[df['rating'] >= df['rating'].quantile(0.75)]

print('=== Sweet Spot Products (topo 25 percent rated) ===')
print(sweet_spot.groupby('style')[['rating', 'price', 'abv']].mean().round(2))
print()
print('Top adjuncts in sweet spot:')
print(sweet_spot['adjunct'].value_counts().head(5))
print()
print('Top flavor profiles in sweet spot: ')
print(sweet_spot['flavor_profile'].value_counts())

=== Sweet Spot Products (topo 25 percent rated) ===
             rating  price    abv
style                            
Braggot        4.57  30.77   9.90
Cyser          4.56  32.64  11.15
Melomel        4.67  32.30  11.04
Metheglin      4.74  35.04  13.42
Traditional    4.63  25.89  12.59

Top adjuncts in sweet spot:
adjunct
Elderflower    16
Raspberry      13
Blackberry     12
Peach          12
Cinnamon       11
Name: count, dtype: int64

Top flavor profiles in sweet spot: 
flavor_profile
Floral    37
Fruity    37
Spicy     18
Citrus     7
Earthy     3
Name: count, dtype: int64


In [16]:
# Seasonal sweet spot - best performing profile per season
print('Best flavor profile per season (by avg rating): ')
df.groupby(['season', 'flavor_profile'])['rating'].mean().groupby('season').idxmax()

Best flavor profile per season (by avg rating): 


season
Autumn    (Autumn, Citrus)
Spring    (Spring, Citrus)
Summer     (Summer, Spicy)
Winter    (Winter, Fruity)
Name: rating, dtype: object

### 7. Export CSV

In [17]:
df.to_csv('iron_honey_mead_data.csv', index=False)
print(f'Exported {len(df)} rows to iron_honey_mead_data.csv')

Exported 400 rows to iron_honey_mead_data.csv


## Summary & Data Source Documentation

**Data Source:** Synthetic dataset of 400 mead products generated programatically. ABV ranges, price points, and rating distributions are modeled on craft beverages industry benchmarks and publicly available meadery pricing data (2022-2024). No public dataset contains sufficient mead-specific entries for meaningful trend analysis

**Filtering Logic:** Products were assigned to one of five mead styles (Traditional, Melomel, Metheglin, Cyser, Barggot) with style_specific ABV andprice distributions. Ten botanical adjuncts were mapped to five flavor profiles (Floral, Spicy, Fruity, Earthy, Citrus). Rating ere influenced by adjunct type and style baseline.

**Key Findings:**

- **Floral profiles (Elderflower, Hibiscus, Lavender)** consistently score the highest consumer rating across all seasons, the safest bet for a new release.
- **Melomel and Metheglin** occupy the highest price brackets and ratings, making them ideal for premium limited releases
- **The ABV sweet spot is 10-14%** products outside this range trend lower in ratings regardless of flavor profile
- **Seasonal pattern:** Floral profiles dominate Spring/Summer, Spicy profiles (Ginger, Cinnamon) perform better in Autumn/Winter
- **Recommended Spring release:** Melomel with Elderflower adjunct, ABV 11-12% priced at $30-38, hits all three sweet spot indicators